# Calculate mean benefit and change in admissions if stroke unit gains MT

Benefit is the difference in proportion of patients who have mRS 0-2 after treatment compared with no treatment. 

For each new MT stroke unit, calculate:
+ the admissions-weighted net benefit over the base scenario for all patients.
+ the total number of patients who benefit more with the new unit than in the base scenario, and
+ the mean benefit for these patients.

__Results__:

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.add_net_benefit import calculate_geo_and_outcomes, \
    calculate_cohort_outcomes_admissions, calculate_benefit
from classes.calc_scenario import build_scenario_parameters, \
    load_scenario_parameters

In [2]:
dir_data = 'data'
dir_output = 'output'
# Reference data:
file_hospitals = 'stroke_hospitals_2025.csv'
# Scenario results:
file_base_geo = 'results_geo_outcomes_base'
file_base_results = 'results_full_base'
# Output:
file_scen_params = 'scenario_parameters_single_unit.csv'
file_benefit_by_team = 'benefit_of_each_team'

In [3]:
# Define transfer loss values based on existing data from the base scenario.

# The following should match the values used to create the base data.
prop_lvo = 0.3
mt_rate = 0.1

col_admissions = 'Admissions_2025_Est'

## Load data

### Hospital info

In [4]:
p = os.path.join(dir_data, file_hospitals)
df_hospitals = pd.read_csv(p, index_col='Postcode')

# Only keep hospitals in England:
df_hospitals = df_hospitals.loc[df_hospitals["Country"] == "England"]

In [5]:
display(df_hospitals.head(3))

,SSNAP name,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
Postcode,,,,,,,,,,,
OX39DU,John Radcliffe Hospital,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
RG15AN,Royal Berkshire Hospital,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
HP112TT,Wycombe General Hospital,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


## 1. Define scenario parameters

In [6]:
# --- Travel ---
# Allow for a travel bias time (minutes) to MT centres
dict_mt_travel_bias = {
    '': 0.1,
    '_bias10min': 10.1
}
s_biases = list(dict_mt_travel_bias.keys())

# --- New MT units ---
# Individual case: have only one of these units at once.
# Get postcode for all hospitals with Use_IVT=1 and Use_MT=0
m = ((df_hospitals['Use_IVT'] == 1) &
     (df_hospitals['Use_MT'] == 0) &
     (df_hospitals['Country'] == 'England'))
units_to_test = df_hospitals.loc[m].index.tolist()

new_unit_options = [[u] for u in units_to_test]

# Scenario names:
scenarios = [f'only_{u}' for u in units_to_test]

Check units to test:

In [7]:
print(len(units_to_test))

print(units_to_test[:5])

73
['RG15AN', 'HP112TT', 'CH21UL', 'L355DR', 'CH495PE']


Gather sets of new units:

### Build scenarios

For each combination of a set of new units and a proportion transferred in drip-and-ship, create scenario parameters.

In [8]:
df_scen_full = build_scenario_parameters(
        scenarios,
        new_unit_options,
        dict_mt_travel_bias
    )
# Save results to file:
p = os.path.join(dir_output, file_scen_params)
df_scen_full.to_csv(p, index=False)
print(f'Saved file: {p}')

Saved file: output/scenario_parameters_single_unit.csv


Load file back in:

In [9]:
p = os.path.join(dir_output, file_scen_params)
df_scen_full = load_scenario_parameters(p)

Check scenario parameters:

In [10]:
display(df_scen_full.head(6))

,scenario_name,scenario_geoprocess_name,new_mt_units,mt_travel_bias_minutes
0,only_RG15AN,only_RG15AN,[RG15AN],0.1
1,only_RG15AN_bias10min,only_RG15AN_bias10min,[RG15AN],10.1
2,only_HP112TT,only_HP112TT,[HP112TT],0.1
3,only_HP112TT_bias10min,only_HP112TT_bias10min,[HP112TT],10.1
4,only_CH21UL,only_CH21UL,[CH21UL],0.1
5,only_CH21UL_bias10min,only_CH21UL_bias10min,[CH21UL],10.1


Pick out unique geography scenario names:

In [11]:
scenarios_geo = df_scen_full['scenario_geoprocess_name'].tolist()

### Cohort parameters

Fixed proportion of patients have LVO (rest have nLVO), and options for proportion transferred for MT are gathered in the same results file.

In [12]:
prop_lvo = 0.3

dict_transferloss = {
    '': 1.0,
    '_transferloss0!67': 0.67
}
options_prop_lvo_dripship_treated_with_mt = list(dict_transferloss.values())
# Column names for transfer rates:
strings_tl = list(dict_transferloss.keys())

# MT rate only affects admissions and transfers, not outcomes:
mt_rate = 0.1

col_admissions = 'Admissions_2025_Est'

## Function to load base scenario results

Base scenario results depend on the travel bias.

Use this function to load in the right data, identify the options for the drip-and-ship transfer loss, and rename columns to mark them as the base case.

Also return dictionaries to link the column names here with those that will be made for each scenario when we change individual stroke units later.

In [13]:
def load_base_scenario_results(s_bias):
    # First load in base scenario results for a comparison:
    p = os.path.join(dir_output, file_base_geo + s_bias + '.csv')
    p2 = os.path.join(dir_output, file_base_results + s_bias + '.csv')
    base_result = pd.concat(
        (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),
        axis='columns'
    )
    
    # Define names of columns with useful data:
    col = 'lvo_drip_ship_ivt_mt_mrs_0-2'
    cols = [c for c in base_result.columns if (
        c.startswith(col) & ('improvement' not in c))]
    transfer_loss_strs = [c.split('_transferloss')[-1] for c in cols]
    options_prop_lvo_dripship_treated_with_mt = sorted(list(set([
        1.0 if t in cols else float(t.replace('!', '.'))
        for t in transfer_loss_strs
    ])))[::-1]  # sort with largest proportion first
    dict_cols_benefit = dict(zip(
        options_prop_lvo_dripship_treated_with_mt, cols))
    
    # Rename base columns:
    cols_base = (['nearest_ivt_unit', 'transfer_unit'] +
                 list(dict_cols_benefit.values()))
    dict_cols_benefit_base = dict(
        [(c, f'{c}_base') for c in cols_base]
    )
    base_result = base_result.rename(columns=dict_cols_benefit_base)
    
    print(dict_cols_benefit)
    
    # Check contents:
    display(base_result[
        [col_admissions] + list(dict_cols_benefit_base.values())
        ].head(5))
    
    # Quick check of mean benefit in the base scenario:
    print("Mean benefit weighted by admissions:")
    for col, col_base in dict_cols_benefit_base.items():
        # Print mean weighted by admissions
        try:
            b = np.average(base_result[col_base],
                           weights=base_result[col_admissions])
            print(f"{col_base:50s} {b:.3f}")
        except TypeError:
            # Probably a string column.
            pass
    return (base_result, options_prop_lvo_dripship_treated_with_mt,
            dict_cols_benefit, dict_cols_benefit_base)

## Function to run for single-unit changes

This runs each scenario in the main scenarios dataframe for this travel time bias.

Don't save the full results to file in this case. Gather summary results in a dictionary.

In [14]:
def run_all_units(
        df_scen_sing,
        options_prop_lvo_dripship_treated_with_mt,
        dict_cols_benefit,
        dict_cols_benefit_base,
        base_result,
    ):
    # Set up dicts to store results:
    hospital_net_benefit = {}
    hospital_admissions_benefitting = {}
    hospital_benefit_for_those_benefitting = {}
    # Separate results for different drip-ship transfer losses:
    for p in options_prop_lvo_dripship_treated_with_mt:
        hospital_net_benefit[p] = {}
        hospital_admissions_benefitting[p] = {}
        hospital_benefit_for_those_benefitting[p] = {}

    for scen_geo_name, df in df_scen_sing.groupby('scenario_geoprocess_name'):
        # These parameters should be the same for all df in group:
        mt_travel_bias = df.iloc[0]['mt_travel_bias_minutes']
        new_mt_units = df.iloc[0]['new_mt_units']

        # Shared geography and outcome results:
        df_geo_outcomes = calculate_geo_and_outcomes(
            mt_travel_bias,
            new_mt_units,
        )
        cols_start = df_geo_outcomes.columns

        results = calculate_cohort_outcomes_admissions(
            df_geo_outcomes,
            options_prop_lvo_dripship_treated_with_mt,
            col_admissions,
            mt_rate,
            prop_lvo,
        )
        cols_new = [c for c in results.columns if c not in cols_start]

        # Bring in base results:
        cols_base = list(dict_cols_benefit_base.values())
        results = pd.concat((results, base_result[cols_base]),
                            axis='columns').reset_index()
        # Only keep useful columns:
        cols_to_keep = ([col_admissions] +
                        list(dict_cols_benefit_base.keys()) +
                        list(dict_cols_benefit_base.values()))
        results = results[cols_to_keep]
    
        # Compare benefit with base:
        for p, col_benefit in dict_cols_benefit.items():
            tup_benefit = calculate_benefit(
                results,
                p,
                col_benefit,
                dict_cols_benefit_base,
                col_admissions,
            )
            # Store these results:
            hospital_net_benefit[p][scen_geo_name] = tup_benefit[0]
            hospital_admissions_benefitting[p][scen_geo_name] = tup_benefit[1]
            hospital_benefit_for_those_benefitting[p][scen_geo_name] = (
                tup_benefit[2])

    return (hospital_net_benefit, hospital_admissions_benefitting,
            hospital_benefit_for_those_benefitting)

Set up for final results dataframe:

In [15]:
def convert_results_lists_to_df(
        hospital_net_benefit,
        hospital_admissions_benefitting,
        hospital_benefit_for_those_benefitting,
        dict_cols_benefit
        ):
    results_dfs = []
    
    for p, col_here in dict_cols_benefit.items():
        s = '' if p == 1.0 else f' ({p} transferred)'.replace('.', '!')
        df = pd.DataFrame({
            'Hospital': list(hospital_net_benefit[p].keys()),
            'Weighted Net Benefit': list(hospital_net_benefit[p].values()),
            'Admission Catchment benefitting': list(
                hospital_admissions_benefitting[p].values()),
            'Mean Benefit for Those Benefitting': list(
                hospital_benefit_for_those_benefitting[p].values()
            )
        })
        df = df.round(5)
        df['Admission Catchment benefitting'] = (
            df['Admission Catchment benefitting'].round(1))
        # Extra calculation:
        best_benefit = df['Weighted Net Benefit'].max()
        df['Relative Benefit'] = (
            (df['Weighted Net Benefit'] / best_benefit).round(3)
            if pd.notna(best_benefit) and best_benefit != 0 else np.nan
        )
    
        # Tidy names.
        # Convert names from "only_Postcode" to "Postcode":
        df['Hospital'] = (
            df['Hospital'].str.split('_', expand=True)[1])
        df = df.set_index('Hospital')
        # Convert column names to show lossiness:
        rename_dict = dict([(c, f'{c}{s}') for c in df.columns])
        df = df.rename(columns=rename_dict)
    
        # Gather results.
        results_dfs.append(df)
    
    # Gather results into dataframe:
    results_df = pd.concat(results_dfs, axis='columns').reset_index()
    
    results_df = results_df.sort_values(by='Weighted Net Benefit',
                                        ascending=False)
    return results_df

## Run all scenarios

In [16]:
def main(
        df_scen_sing,
        s_bias
    ):
    # Pick out the travel time bias based on the input string:
    if len(s_bias) > 0:
        m = df_scen_sing['scenario_geoprocess_name'].str.contains(s_bias)
    else:
        m = ~df_scen_sing['scenario_geoprocess_name'].str.contains('bias')
    bias = df_scen_sing.loc[m, 'mt_travel_bias_minutes'].tolist()[0]
    print(s_bias, bias)
    
    # Keep only the scenarios with the right travel time bias:
    df_scen_to_run = df_scen_sing[df_scen_sing['mt_travel_bias_minutes'] == bias]
    n_scenarios_to_run = len(df_scen_to_run)
    print('Scenarios to run:', n_scenarios_to_run)
    
    (base_result, options_prop_lvo_dripship_treated_with_mt,
     dict_cols_benefit, dict_cols_benefit_base) = (
         load_base_scenario_results(s_bias))
    print('Transfer loss options:', options_prop_lvo_dripship_treated_with_mt)
    
    (hospital_net_benefit, hospital_admissions_benefitting,
     hospital_benefit_for_those_benefitting) = run_all_units(
            df_scen_to_run,
            options_prop_lvo_dripship_treated_with_mt,
            dict_cols_benefit,
            dict_cols_benefit_base,
            base_result,
        )

    results_df = convert_results_lists_to_df(
        hospital_net_benefit,
        hospital_admissions_benefitting,
        hospital_benefit_for_those_benefitting,
        dict_cols_benefit
        )
    print('')
    return results_df

In [17]:
for s_bias in ['', '_bias10min']:
    results_df = main(df_scen_full, s_bias)

    # Merge in unit info:
    results_df = pd.merge(
        df_hospitals.reset_index()[['Postcode', 'SSNAP name', 'Hospital_City']],
        results_df,
        how='right',
        left_on='Postcode',
        right_on='Hospital',
    ).set_index('SSNAP name').drop('Postcode', axis='columns')

    # Save results
    p = os.path.join(dir_output, file_benefit_by_team + s_bias + '.csv')
    results_df.to_csv(p, index=True)

 0.1
Scenarios to run: 73
{1.0: 'lvo_drip_ship_ivt_mt_mrs_0-2', 0.67: 'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67'}


/tmp/ipykernel_49957/2947273500.py:6: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),


,Admissions_2025_Est,nearest_ivt_unit_base,transfer_unit_base,lvo_drip_ship_ivt_mt_mrs_0-2_base,lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base
LSOA,,,,,
ADUR001A,1.49403,BN25BE,BN25BE,0.429,0.429
ADUR001B,1.86762,BN25BE,BN25BE,0.429,0.429
ADUR001C,2.24110,BN112DH,BN25BE,0.376,0.359
ADUR001D,2.24110,BN25BE,BN25BE,0.429,0.429
ADUR001E,4.10862,BN112DH,BN25BE,0.377,0.360


Mean benefit weighted by admissions:
lvo_drip_ship_ivt_mt_mrs_0-2_base                  0.389
lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base 0.379
Transfer loss options: [1.0, 0.67]
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
Th

/tmp/ipykernel_49957/2947273500.py:6: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),


,Admissions_2025_Est,nearest_ivt_unit_base,transfer_unit_base,lvo_drip_ship_ivt_mt_mrs_0-2_base,lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base
LSOA,,,,,
ADUR001A,1.49403,BN25BE,BN25BE,0.429,0.429
ADUR001B,1.86762,BN25BE,BN25BE,0.429,0.429
ADUR001C,2.24110,BN25BE,BN25BE,0.428,0.428
ADUR001D,2.24110,BN25BE,BN25BE,0.429,0.429
ADUR001E,4.10862,BN25BE,BN25BE,0.428,0.428


Mean benefit weighted by admissions:
lvo_drip_ship_ivt_mt_mrs_0-2_base                  0.400
lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base 0.394
Transfer loss options: [1.0, 0.67]
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
Th

## Check results

In [18]:
display(results_df.head(10))

,Hospital_City,Hospital,Weighted Net Benefit,Admission Catchment benefitting,Mean Benefit for Those Benefitting,Relative Benefit,Weighted Net Benefit (0!67 transferred),Admission Catchment benefitting (0!67 transferred),Mean Benefit for Those Benefitting (0!67 transferred),Relative Benefit (0!67 transferred)
SSNAP name,,,,,,,,,,
Frimley Park Hospital,Frimley Park,GU167UJ,0.00119,2690.9,0.04001,1.000,0.00145,2690.9,0.04834,1.000
Luton and Dunstable Hospital,Luton and Dunstable,LU40DZ,0.00116,3834.4,0.02741,0.975,0.00138,3834.4,0.03234,0.952
Northwick Park Hospital HASU,Harrow,HA13UJ,0.00113,5950.2,0.01726,0.950,0.00144,5906.9,0.02192,0.993
Watford General Hospital,Watford,WD180HB,0.00112,4268.7,0.02379,0.941,0.00139,4268.7,0.02924,0.959
Musgrove Park Hospital,Taunton,TA15DA,0.00103,2481.7,0.03777,0.866,0.00113,2532.5,0.04012,0.779
Royal Berkshire Hospital,Reading,RG15AN,0.00097,3485.2,0.02513,0.815,0.00115,3376.7,0.03063,0.793
University Hospitals Dorset Stroke Service,Bournemouth,BH77DW,0.00097,2106.6,0.04170,0.815,0.00120,2106.6,0.05090,0.828
Northampton General Hospital,Northampton,NN15BD,0.00094,3295.5,0.02586,0.790,0.00110,3266.7,0.03024,0.759
Darent Valley Hospital,Dartford,DA28DA,0.00093,2623.4,0.03234,0.782,0.00121,2623.4,0.04133,0.834
